In [ ]:
import os
import subprocess
import sys
from pathlib import Path

print("====== STEP 1: CLONING BENCHMARK AND AF-CLIP ======")

benchmark_dir = Path("/kaggle/working/Natural-Corruption-Robustness")
afclip_dir = Path("/kaggle/working/AF-CLIP")
BENCHMARK_REPOSITORY = "Parsagh05/Natural-Corruption-Robustness"
benchmark_url = f"https://github.com/{BENCHMARK_REPOSITORY}.git"

if not benchmark_dir.exists():
    subprocess.run(
        ["git", "clone", benchmark_url, str(benchmark_dir)], check=True
    )
else:
    subprocess.run(
        ["git", "-C", str(benchmark_dir), "pull", "--ff-only"],
        check=True,
    )

if not afclip_dir.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/Faustinaqq/AF-CLIP.git",
            str(afclip_dir),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(afclip_dir), "pull", "--ff-only"],
        check=True,
    )

required_source_files = [
    afclip_dir / "clip" / "clip.py",
    afclip_dir / "clip" / "model.py",
    afclip_dir / "clip" / "adaptor.py",
]
missing_source_files = [path for path in required_source_files if not path.exists()]
if missing_source_files:
    raise FileNotFoundError(
        "AF-CLIP clone is incomplete; missing: "
        + ", ".join(str(path) for path in missing_source_files)
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ftfy",
        "regex",
        "tqdm",
        "einops",
        "scipy",
        "opencv-python",
        "scikit-learn",
        "scikit-image",
        "pandas",
    ],
    check=True,
)

os.environ["AFCLIP_ROOT"] = str(afclip_dir)
print(f"Benchmark repository: {benchmark_dir}")
print(f"Official AF-CLIP:      {afclip_dir}")
print("Environment ready.")


In [ ]:
import gc
import os
import sys
from pathlib import Path
import torch

AFCLIP_ROOT = Path("/kaggle/working/AF-CLIP")
HARNESS_ROOT = Path("/kaggle/working/Natural-Corruption-Robustness")
if str(HARNESS_ROOT) not in sys.path:
    sys.path.insert(0, str(HARNESS_ROOT))
os.environ["AFCLIP_ROOT"] = str(AFCLIP_ROOT)

from harness.runner import run_evaluation

# Choose exactly one evaluation target.
# DATASET_NAME = "mvtec"
DATASET_NAME = "visa"
MODEL_NAME = "AF-CLIP"

# AF-CLIP's official zero-shot protocol is cross-dataset: weights learned
# on VisA evaluate MVTec, and weights learned on MVTec evaluate VisA.
DATASET_NAME = DATASET_NAME.lower().strip()
if DATASET_NAME not in {"mvtec", "visa"}:
    raise ValueError("DATASET_NAME must be either 'mvtec' or 'visa'.")
IS_MVTEC = DATASET_NAME == "mvtec"
WEIGHT_DATASET = "visa" if IS_MVTEC else "mvtec"
SELECTED_DATASET = "MVTec AD" if IS_MVTEC else "VisA"

# True uses the balanced natural-noise categories and their persistent CSV
# assignment plan, including paired image/mask geometric transforms.
USE_CATEGORIZED_CORRUPTIONS = True
CATEGORIZED_CORRUPTION_SEED = 123
UNCATEGORIZED_CORRUPTION_TYPES = [
    "gaussian_noise",
    "shot_noise",
    "impulse_noise",
    "defocus_blur",
    "motion_blur",
    "zoom_blur",
    "brightness",
    "contrast",
]
CATEGORIZED_CORRUPTION_TYPES = [
    "noise",
    "blur",
    "photometric",
    "geometric",
]
CORRUPTION_TYPES = (
    CATEGORIZED_CORRUPTION_TYPES
    if USE_CATEGORIZED_CORRUPTIONS
    else UNCATEGORIZED_CORRUPTION_TYPES
)
SEVERITY_LEVELS = [1, 2, 3, 4]
BATCH_SIZE = 2
CORRUPTION_CACHE_ROOT = None
CORRUPTION_CACHE_FORMAT = "png"

MVTEC_PATH = (
    "/kaggle/input/datasets/alirezasalehy/mvtec-ad/"
    "mvtec_anomaly_detection"
)
VISA_PATH = "/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922"
OUTPUT_ROOT = "/kaggle/working/outputs"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

plan_name = (
    "mvtec_corruption_plan.csv"
    if IS_MVTEC
    else "visa_corruption_plan.csv"
)
CORRUPTION_PLAN = HARNESS_ROOT / plan_name
if USE_CATEGORIZED_CORRUPTIONS and not CORRUPTION_PLAN.exists():
    raise FileNotFoundError(
        f"Categorized corruption plan not found: {CORRUPTION_PLAN}"
    )

# The learned AF-CLIP prompt/adaptor weights are included in the official
# GitHub clone. The base OpenAI ViT-L/14@336px weight is separate. Reuse a
# Kaggle input when available; otherwise AF-CLIP downloads and verifies it.
AFCLIP_WEIGHT_DIR = AFCLIP_ROOT / "weight"
AFCLIP_PROMPT_WEIGHT = AFCLIP_WEIGHT_DIR / f"{WEIGHT_DATASET}_prompt.pt"
AFCLIP_ADAPTOR_WEIGHT = AFCLIP_WEIGHT_DIR / f"{WEIGHT_DATASET}_adaptor.pt"
for weight_path in (AFCLIP_PROMPT_WEIGHT, AFCLIP_ADAPTOR_WEIGHT):
    if not weight_path.exists():
        raise FileNotFoundError(
            f"AF-CLIP weight missing from the GitHub clone: {weight_path}"
        )

clip_weight_candidates = [
    AFCLIP_ROOT / "download" / "clip" / "ViT-L-14-336px.pt",
]
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    clip_weight_candidates.extend(kaggle_input.rglob("ViT-L-14-336px.pt"))
AFCLIP_CLIP_WEIGHT = next(
    (str(path) for path in clip_weight_candidates if path.is_file()),
    "",
)

model_kwargs = {
    MODEL_NAME: {
        "afclip_root": str(AFCLIP_ROOT),
        "checkpoint_path": str(AFCLIP_WEIGHT_DIR),
        "weight_dataset": WEIGHT_DATASET,
        "clip_weight_path": AFCLIP_CLIP_WEIGHT,
        "clip_download_dir": str(AFCLIP_ROOT / "download" / "clip"),
        "clip_model_name": "ViT-L/14@336px",
        "image_size": 518,
        "prompt_len": 12,
        "feature_layers": [6, 12, 18, 24],
        "memory_layers": [6, 12, 18, 24],
        "alpha": 0.1,
    }
}

print("LAUNCHING AF-CLIP ROBUSTNESS BENCHMARK")
print(f"Evaluation target: {SELECTED_DATASET}")
print(f"AF-CLIP weights:  trained on {WEIGHT_DATASET}")
print(f"Prompt weight:    {AFCLIP_PROMPT_WEIGHT}")
print(f"Adaptor weight:   {AFCLIP_ADAPTOR_WEIGHT}")
print(f"CLIP backbone:    {AFCLIP_CLIP_WEIGHT or 'automatic download'}")
print(f"Corruptions:      {CORRUPTION_TYPES} @ {SEVERITY_LEVELS}")
print(f"Categorized:      {USE_CATEGORIZED_CORRUPTIONS}")
print(f"Plan:             {CORRUPTION_PLAN}")
print(f"Device/batch:     {DEVICE} / {BATCH_SIZE}")
print(f"Outputs:          {OUTPUT_ROOT}")

run_evaluation(
    mvtec_root=MVTEC_PATH if IS_MVTEC else None,
    visa_root=None if IS_MVTEC else VISA_PATH,
    output_root=OUTPUT_ROOT,
    models=[MODEL_NAME],
    model_kwargs=model_kwargs,
    device=DEVICE,
    dataset=DATASET_NAME,
    corruption_types=CORRUPTION_TYPES,
    severity_levels=SEVERITY_LEVELS,
    batch_size=BATCH_SIZE,
    corruption_cache_root=CORRUPTION_CACHE_ROOT,
    corruption_cache_format=CORRUPTION_CACHE_FORMAT,
    categorized_corruptions=USE_CATEGORIZED_CORRUPTIONS,
    categorized_corruption_plans={
        DATASET_NAME: str(CORRUPTION_PLAN)
    },
    corruption_seed=(
        CATEGORIZED_CORRUPTION_SEED
        if USE_CATEGORIZED_CORRUPTIONS
        else None
    ),
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"AF-CLIP evaluation complete. Outputs: {OUTPUT_ROOT}")
